In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.chdir('/content/drive/MyDrive')

!mkdir -p olist-customer-intelligence/{data/raw,data/processed,notebooks,sql,src,models,reports,app/pages,docs}
os.chdir('/content/drive/MyDrive/M5-forecasting')
!touch src/__init__.py
print("Now in:", os.getcwd())

Now in: /content/drive/MyDrive/M5-forecasting


In [5]:
%%writefile src/models/deep_model.py
"""Global N-BEATS forecasting via Darts for M5 leaf-level series."""
import numpy as np
import pandas as pd


def make_timeseries(long_df, series_ids, id_col="id", time_col="date",
                    value_col="sales"):
    """Convert selected series in a long frame to a list of Darts TimeSeries."""
    from darts import TimeSeries
    series_list = []
    kept_ids = []
    for sid in series_ids:
        g = long_df[long_df[id_col] == sid].sort_values(time_col)
        if len(g) < 60:          # need enough history
            continue
        ts = TimeSeries.from_dataframe(
            g, time_col=time_col, value_cols=value_col, freq="D"
        )
        series_list.append(ts)
        kept_ids.append(sid)
    return series_list, kept_ids


def build_nbeats(input_chunk=56, output_chunk=28, quantiles=(0.1, 0.5, 0.9)):
    from darts.models import NBEATSModel
    from darts.utils.likelihood_models import QuantileRegression
    model = NBEATSModel(
        input_chunk_length=input_chunk,
        output_chunk_length=output_chunk,
        num_stacks=10, num_blocks=1, num_layers=4,
        layer_widths=256,
        n_epochs=15, batch_size=512,
        likelihood=QuantileRegression(quantiles=list(quantiles)),
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1,
                           "enable_progress_bar": True},
    )
    return model

Overwriting src/models/deep_model.py


# Global N-BEATS Training Pipeline

### 1. Data Preparation
- Sample **1500 Item × Store products**
- Convert each product into a **Darts TimeSeries**
- Each TimeSeries contains approximately **674 training days**

↓

### 2. Sliding Window Generation
For every TimeSeries:
- Input  = Previous **56 days**
- Target = Next **28 days**

Example:

Input:
Day 1 ─────────────────────────────► Day 56

Target:
Day 57 ────────────────────────────► Day 84

This creates:
- ~591 windows per product
- ~900,000 (Input, Target) training samples

↓

### 3. Shuffle & Batching
- Shuffle all windows every epoch
- Create batches of **512 windows**
- Each batch contains windows from many different products

↓

### 4. ONE Global N-BEATS Model

```
                Input (56 values)
                      │
                      ▼
            Stack 1 (4 Dense Layers)
                 256 → 256 → 256 → 256
                      │
                      ▼
            Stack 2 (4 Dense Layers)
                      │
                      ▼
                     ...
                      │
                      ▼
            Stack 10 (4 Dense Layers)
                      │
                      ▼
       Output: 28-day Forecast (P10, P50, P90)
```

**Important:**
- There is **ONE** neural network.
- There are **NOT 1500 neural networks.**
- All products share the same weights.

↓

### 5. Training One Batch

```
512 Windows
      │
      ▼
Forward Pass
      │
      ▼
512 Predictions
      │
      ▼
Compare with True Targets
      │
      ▼
Average Pinball Loss
      │
      ▼
Backpropagation
(Output → Stack10 → ... → Stack1)
      │
      ▼
Adam Optimizer updates ALL weights
      │
      ▼
Next Batch
```

↓

### 6. One Epoch

- ~900,000 windows
- Batch size = 512
- ≈1758 batches per epoch
- Each batch performs:
  - Forward pass
  - Loss computation
  - Backpropagation
  - Weight update

Repeat for **15 epochs**

---

## Final Model

After training:

```
Last 56 Days of ANY Product
            │
            ▼
     SAME N-BEATS MODEL
            │
            ▼
28-Day Forecast
(P10, P50, P90)
```

### Key Takeaways

- One global model learns from all products.
- Training examples are **windows**, not products.
- Every batch updates the **same shared weights**.
- The trained model can forecast **any product** using its last 56 days of sales.

## Deep Learning (N-BEATS via Darts)

**Model:** Global N-BEATS, quantile likelihood (P10/P50/P90), trained on 1,500 representative item-level series (stratified across store×category). 2.2M params, 15 epochs.

**Environment note:** Darts/PyTorch version conflicts made Colab unworkable (numpy/torch/torchvision lock). Trained **locally on MacBook (Apple Silicon, MPS)** via `run_deep_local.py`; forecasts + model uploaded to Drive. Full-scale training across all 30,490 series is a straightforward scale-up — log in limitations.md.

**Setup:**
- input_chunk=56, output_chunk=28 (matches horizon)
- float32 + precision "32-true" (MPS doesn't support float64)
- ~1.5 min/epoch on MPS, ~20 min total

**Results (28-day test, 1,500-series subset):**
- Pinball — P10: 0.118, P50: 0.430, P90: 0.292
- P50 RMSE: 1.762
- **Coverage: 0.786** vs 0.80 target → well calibrated (tighter than LightGBM's 0.881)

**Caveat:** subset-level metrics; cross-model verdict comes from WRMSSE (Day 25), not these RMSEs.

**Saved:** `forecasts_deep.parquet`, `models/deep_model.pt`, `run_deep_local.py`